In [1]:
%pip install -U pandas numpy requests xlrd tqdm

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ============================================================
# NOAA ATLAS 14
# DOWNLOAD PDS-BASED PRECIPITATION-FREQUENCY ESTIMATES
# FOR ALL ALABAMA STATIONS LISTED IN al.xls
# ============================================================

from __future__ import annotations

import csv
import io
import re
import time
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests

from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry


# ============================================================
# 1. USER SETTINGS
# ============================================================

# The al.xls file should be in the same folder as this notebook.
# Alternatively, replace this with the complete file path.
STATION_FILE = Path.cwd() / "al.xls"

# Output directory.
OUTPUT_ROOT = Path.cwd() / "NOAA_Atlas14_Alabama_PDS"

# NOAA station-list fallback URL.
# Used only if al.xls is not found locally.
STATION_LIST_URL = (
    "https://hdsc.nws.noaa.gov/pub/hdsc/data/xlslists/al.xls"
)

# NOAA Atlas 14 precipitation-frequency CSV endpoint.
NOAA_API_URL = (
    "https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text.csv"
)

# Download settings.
DATA_TYPE = "depth"
UNITS = "metric"
SERIES = "pds"

# Pause between NOAA requests.
SECONDS_BETWEEN_REQUESTS = 0.75

# HTTP timeout settings.
CONNECT_TIMEOUT_SECONDS = 30
READ_TIMEOUT_SECONDS = 120

# False means valid files already downloaded will be reused.
# True forces every station to be downloaded again.
FORCE_REDOWNLOAD = False

# None processes every unique station.
#
# For a short test, temporarily use:
# MAX_STATIONS = 3
MAX_STATIONS: int | None = None

USER_AGENT = (
    "Alabama-StageIV-Intense-Rainfall-Research/1.0 "
    "(academic NOAA Atlas 14 analysis)"
)


# ============================================================
# 2. CREATE OUTPUT DIRECTORIES
# ============================================================

RAW_DIR = OUTPUT_ROOT / "raw_noaa_responses"
TABLE_DIR = OUTPUT_ROOT / "tables"
ERROR_DIR = OUTPUT_ROOT / "errors"

for folder in [
    OUTPUT_ROOT,
    RAW_DIR,
    TABLE_DIR,
    ERROR_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Current notebook folder:")
print(Path.cwd())

print("\nStation file:")
print(STATION_FILE)

print("\nOutput folder:")
print(OUTPUT_ROOT)


# ============================================================
# 3. CREATE HTTP SESSION WITH AUTOMATIC RETRIES
# ============================================================

def build_session() -> requests.Session:
    """
    Create a requests session that automatically retries
    temporary NOAA server and connection errors.
    """

    retry_strategy = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=1.5,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=2,
        pool_maxsize=2,
    )

    session = requests.Session()

    session.mount(
        "https://",
        adapter,
    )

    session.mount(
        "http://",
        adapter,
    )

    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": "text/csv,text/plain,*/*",
        }
    )

    return session


SESSION = build_session()


# ============================================================
# 4. DOWNLOAD A FILE SAFELY
# ============================================================

def download_binary_file(
    url: str,
    destination: Path,
    session: requests.Session,
) -> None:
    """
    Download a binary file and save it using a temporary file.
    """

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_file = destination.with_suffix(
        destination.suffix + ".part"
    )

    response = session.get(
        url,
        timeout=(
            CONNECT_TIMEOUT_SECONDS,
            READ_TIMEOUT_SECONDS,
        ),
    )

    response.raise_for_status()

    temporary_file.write_bytes(
        response.content
    )

    temporary_file.replace(
        destination
    )


# Download al.xls only when it is not already in the notebook folder.
if not STATION_FILE.exists():

    print(
        "\nal.xls was not found locally. "
        "Downloading it from NOAA..."
    )

    download_binary_file(
        url=STATION_LIST_URL,
        destination=STATION_FILE,
        session=SESSION,
    )


if not STATION_FILE.exists():

    raise FileNotFoundError(
        "The Alabama station workbook could not be found:\n"
        f"{STATION_FILE}"
    )


# ============================================================
# 5. READ AND CLEAN THE NOAA ALABAMA STATION WORKBOOK
# ============================================================

def clean_column_names(
    columns,
) -> list[str]:
    """
    Remove leading, trailing, and repeated spaces
    from Excel column names.
    """

    cleaned = []

    for column in columns:

        name = str(column).strip()

        name = re.sub(
            r"\s+",
            " ",
            name,
        )

        cleaned.append(name)

    return cleaned


station_rows = pd.read_excel(
    STATION_FILE,
    sheet_name=0,
    engine="xlrd",
    dtype={
        "State": str,
        "Station name": str,
        "Station ID": str,
        "Post-merge station ID": str,
        "Co-located station ID": str,
        "Base duration": str,
        "Source of data": str,
        "Period of record": str,
    },
)

station_rows.columns = clean_column_names(
    station_rows.columns
)

print("\nColumns found in al.xls:")

for column in station_rows.columns:
    print(" -", column)


required_columns = {
    "State",
    "Station name",
    "Station ID",
    "Latitude",
    "Longitude",
}

missing_columns = sorted(
    required_columns.difference(
        station_rows.columns
    )
)

if missing_columns:

    raise ValueError(
        "The workbook is missing required columns: "
        + ", ".join(missing_columns)
    )


# Clean station identifiers and names.
station_rows["Station ID"] = (
    station_rows["Station ID"]
    .astype(str)
    .str.strip()
)

station_rows["Station name"] = (
    station_rows["Station name"]
    .astype(str)
    .str.strip()
)


# Convert coordinates to numeric.
station_rows["Latitude"] = pd.to_numeric(
    station_rows["Latitude"],
    errors="coerce",
)

station_rows["Longitude"] = pd.to_numeric(
    station_rows["Longitude"],
    errors="coerce",
)


# Remove records without coordinates.
missing_coordinate_rows = station_rows.loc[
    station_rows["Latitude"].isna()
    | station_rows["Longitude"].isna()
].copy()

if not missing_coordinate_rows.empty:

    missing_coordinate_rows.to_csv(
        TABLE_DIR / "stations_missing_coordinates.csv",
        index=False,
    )

    warnings.warn(
        f"{len(missing_coordinate_rows)} workbook rows "
        "do not have usable coordinates and will be skipped."
    )


station_rows = station_rows.loc[
    station_rows["Latitude"].notna()
    & station_rows["Longitude"].notna()
].copy()


# ============================================================
# 6. REDUCE THE WORKBOOK TO UNIQUE STATIONS
# ============================================================

def join_unique_text(
    values: pd.Series,
) -> str:
    """
    Join unique, non-empty text entries while preserving order.
    """

    output = []

    for value in values:

        if pd.isna(value):
            continue

        text = str(value).strip()

        if (
            text
            and text.lower() != "nan"
            and text not in output
        ):
            output.append(text)

    return "; ".join(output)


# Identify station IDs that have conflicting coordinates.
coordinate_check = (
    station_rows
    .groupby("Station ID")
    .agg(
        latitude_count=(
            "Latitude",
            "nunique",
        ),
        longitude_count=(
            "Longitude",
            "nunique",
        ),
    )
    .reset_index()
)

coordinate_conflicts = coordinate_check.loc[
    (coordinate_check["latitude_count"] > 1)
    | (coordinate_check["longitude_count"] > 1)
].copy()

coordinate_conflicts.to_csv(
    TABLE_DIR / "station_coordinate_conflicts.csv",
    index=False,
)

if not coordinate_conflicts.empty:

    warnings.warn(
        f"{len(coordinate_conflicts)} station IDs have more than "
        "one coordinate. The first coordinate will be used."
    )


aggregation_rules = {
    "State": "first",
    "Station name": "first",
    "Latitude": "first",
    "Longitude": "first",
}

optional_text_columns = [
    "Post-merge station ID",
    "Co-located station ID",
    "Base duration",
    "Source of data",
    "Period of record",
]

optional_numeric_columns = [
    "Elevation (ft)",
]

for column in optional_text_columns:

    if column in station_rows.columns:
        aggregation_rules[column] = join_unique_text


for column in optional_numeric_columns:

    if column in station_rows.columns:
        aggregation_rules[column] = "first"


stations = (
    station_rows
    .groupby(
        "Station ID",
        as_index=False,
    )
    .agg(aggregation_rules)
    .sort_values("Station ID")
    .reset_index(drop=True)
)


if MAX_STATIONS is not None:

    stations = (
        stations
        .head(MAX_STATIONS)
        .copy()
    )


station_inventory_path = (
    TABLE_DIR
    / "alabama_unique_station_inventory.csv"
)

stations.to_csv(
    station_inventory_path,
    index=False,
)


print("\nStation inventory summary")
print("-------------------------")
print(f"Workbook records: {len(station_rows):,}")
print(f"Unique stations: {len(stations):,}")

display(
    stations.head(10)
)


# ============================================================
# 7. NOAA RESPONSE PARSING FUNCTIONS
# ============================================================

DURATION_PATTERN = re.compile(
    r"^\d+(?:\.\d+)?-(?:min|hr|day):?$",
    flags=re.IGNORECASE,
)


def safe_filename(
    value: str,
) -> str:
    """
    Convert text into a filename that is safe on Windows.
    """

    value = str(value).strip()

    value = re.sub(
        r'[<>:"/\\|?*]+',
        "_",
        value,
    )

    value = re.sub(
        r"\s+",
        "_",
        value,
    )

    return value.strip("._") or "unknown"


def is_valid_noaa_response(
    text: str,
) -> bool:
    """
    Check whether a response contains all three NOAA
    precipitation-frequency estimate sections.
    """

    upper_text = text.upper()

    required_phrases = [
        "PRECIPITATION FREQUENCY ESTIMATES",
        "BY DURATION FOR ARI",
        "UPPER BOUND OF 90% CONFIDENCE INTERVAL",
        "LOWER BOUND OF 90% CONFIDENCE INTERVAL",
    ]

    return all(
        phrase in upper_text
        for phrase in required_phrases
    )


def parse_number(
    value: str,
) -> float:
    """
    Convert a NOAA CSV cell to a floating-point number.
    """

    text = str(value).strip()

    if text == "":
        return np.nan

    text = text.replace(
        ",",
        "",
    )

    if text.upper() in {
        "NA",
        "N/A",
        "NAN",
        "NONE",
        "-",
        "--",
    }:
        return np.nan

    text = text.strip(
        "()[]{}"
    )

    try:
        return float(text)

    except ValueError:
        return np.nan


def parse_ari_values(
    cells: list[str],
) -> list[float]:
    """
    Extract average recurrence intervals from a NOAA heading.
    """

    ari_values = []

    for cell in cells[1:]:

        value = parse_number(cell)

        if np.isfinite(value):
            ari_values.append(
                float(value)
            )

    return ari_values


def parse_noaa_frequency_csv(
    text: str,
    station_metadata: dict[str, Any],
) -> pd.DataFrame:
    """
    Convert one NOAA response into a tidy table.

    One output record represents:

    station × estimate type × duration × recurrence interval
    """

    if not is_valid_noaa_response(text):

        preview = (
            text[:500]
            .replace("\n", " | ")
        )

        raise ValueError(
            "The response does not contain the expected NOAA "
            f"frequency tables. Preview: {preview}"
        )


    csv_reader = csv.reader(
        io.StringIO(
            text.lstrip("\ufeff")
        )
    )

    current_estimate_type = None
    current_ari_values = []

    output_rows = []


    for raw_row in csv_reader:

        cells = [
            str(cell).strip()
            for cell in raw_row
        ]

        if not cells:
            continue


        joined_upper = " ".join(
            cells
        ).upper()


        # Upper confidence interval must be checked first.
        if (
            "PRECIPITATION FREQUENCY ESTIMATES"
            in joined_upper
            and "AT UPPER BOUND"
            in joined_upper
        ):

            current_estimate_type = "upper_90"

            current_ari_values = parse_ari_values(
                cells
            )

            continue


        # Lower confidence interval.
        if (
            "PRECIPITATION FREQUENCY ESTIMATES"
            in joined_upper
            and "AT LOWER BOUND"
            in joined_upper
        ):

            current_estimate_type = "lower_90"

            current_ari_values = parse_ari_values(
                cells
            )

            continue


        # Main estimate table.
        if (
            "PRECIPITATION FREQUENCY ESTIMATES"
            in joined_upper
            and "BY DURATION FOR ARI"
            in joined_upper
        ):

            current_estimate_type = "estimate"

            current_ari_values = parse_ari_values(
                cells
            )

            continue


        if current_estimate_type is None:
            continue


        duration = (
            cells[0]
            .rstrip(":")
            .strip()
        )


        if not DURATION_PATTERN.match(duration):
            continue


        values = [
            parse_number(cell)
            for cell in cells[
                1:
                1 + len(current_ari_values)
            ]
        ]


        if len(values) != len(current_ari_values):

            raise ValueError(
                f"Duration {duration!r} contains "
                f"{len(values)} values, but "
                f"{len(current_ari_values)} ARIs were found."
            )


        for ari_years, precipitation_value in zip(
            current_ari_values,
            values,
        ):

            output_row = dict(
                station_metadata
            )

            output_row.update(
                {
                    "series": SERIES,
                    "data_type": DATA_TYPE,
                    "units_requested": UNITS,
                    "estimate_type": current_estimate_type,
                    "duration": duration,
                    "ari_years": ari_years,
                    "precipitation_value": precipitation_value,
                }
            )

            output_rows.append(
                output_row
            )


    parsed = pd.DataFrame(
        output_rows
    )


    if parsed.empty:

        raise ValueError(
            "The NOAA response was recognized, but no duration "
            "and recurrence-interval records were parsed."
        )


    expected_estimate_types = {
        "estimate",
        "upper_90",
        "lower_90",
    }

    found_estimate_types = set(
        parsed["estimate_type"]
        .dropna()
        .unique()
    )


    if found_estimate_types != expected_estimate_types:

        raise ValueError(
            "Expected estimate types "
            f"{sorted(expected_estimate_types)}, but found "
            f"{sorted(found_estimate_types)}."
        )


    return parsed


# ============================================================
# 8. REQUEST ONE STATION FROM NOAA
# ============================================================

def request_station(
    station_metadata: dict[str, Any],
) -> tuple[str, str]:
    """
    Download the NOAA Atlas 14 CSV response for one coordinate.
    """

    parameters = {
        "lat": (
            f"{station_metadata['latitude']:.6f}"
        ),
        "lon": (
            f"{station_metadata['longitude']:.6f}"
        ),
        "data": DATA_TYPE,
        "units": UNITS,
        "series": SERIES,
    }


    response = SESSION.get(
        NOAA_API_URL,
        params=parameters,
        timeout=(
            CONNECT_TIMEOUT_SECONDS,
            READ_TIMEOUT_SECONDS,
        ),
    )


    response.raise_for_status()

    response_text = response.text


    if not is_valid_noaa_response(
        response_text
    ):

        preview = (
            response_text[:500]
            .replace("\n", " | ")
        )

        raise ValueError(
            "NOAA returned an unexpected response. "
            f"Request URL: {response.url}. "
            f"Response preview: {preview}"
        )


    return (
        response_text,
        response.url,
    )


# ============================================================
# 9. CONVERT A STATION ROW TO OUTPUT METADATA
# ============================================================

def station_metadata_from_row(
    row: pd.Series,
) -> dict[str, Any]:
    """
    Create standardized metadata for one station.
    """

    metadata = {
        "station_id": str(
            row["Station ID"]
        ),
        "station_name": str(
            row["Station name"]
        ),
        "state": str(
            row["State"]
        ),
        "latitude": float(
            row["Latitude"]
        ),
        "longitude": float(
            row["Longitude"]
        ),
    }


    optional_column_map = {
        "Post-merge station ID": "post_merge_station_id",
        "Co-located station ID": "co_located_station_id",
        "Base duration": "base_duration",
        "Source of data": "source_of_data",
        "Elevation (ft)": "elevation_ft",
        "Period of record": "period_of_record",
    }


    for input_column, output_column in optional_column_map.items():

        if input_column in row.index:

            metadata[output_column] = (
                row[input_column]
            )


    return metadata


# ============================================================
# 10. DOWNLOAD AND PARSE EVERY UNIQUE STATION
# ============================================================

all_station_tables = []
download_log = []


for _, station_row in tqdm(
    stations.iterrows(),
    total=len(stations),
    desc="Downloading NOAA Atlas 14 PDS estimates",
):

    metadata = station_metadata_from_row(
        station_row
    )

    station_id = metadata["station_id"]
    station_name = metadata["station_name"]


    # Use the station ID for short, Windows-safe filenames.
    file_stem = safe_filename(
        station_id
    )

    raw_file = (
        RAW_DIR
        / f"{file_stem}_pds_frequency.csv"
    )

    error_file = (
        ERROR_DIR
        / f"{file_stem}.error.txt"
    )


    status_record = {
        "station_id": station_id,
        "station_name": station_name,
        "latitude": metadata["latitude"],
        "longitude": metadata["longitude"],
        "raw_file": str(raw_file),
        "status": None,
        "request_url": None,
        "rows_parsed": 0,
        "error": None,
    }


    try:

        use_existing_file = (
            raw_file.exists()
            and not FORCE_REDOWNLOAD
        )


        if use_existing_file:

            raw_text = raw_file.read_text(
                encoding="utf-8",
                errors="replace",
            )

            # Re-download invalid or incomplete existing files.
            if not is_valid_noaa_response(
                raw_text
            ):
                use_existing_file = False


        if use_existing_file:

            status_record["status"] = (
                "reused_existing"
            )


        else:

            raw_text, request_url = request_station(
                metadata
            )

            status_record["request_url"] = (
                request_url
            )

            temporary_file = raw_file.with_suffix(
                ".csv.part"
            )

            temporary_file.write_text(
                raw_text,
                encoding="utf-8",
            )

            temporary_file.replace(
                raw_file
            )

            status_record["status"] = (
                "downloaded"
            )

            time.sleep(
                SECONDS_BETWEEN_REQUESTS
            )


        parsed_station = parse_noaa_frequency_csv(
            raw_text,
            metadata,
        )

        all_station_tables.append(
            parsed_station
        )

        status_record["rows_parsed"] = len(
            parsed_station
        )


        if error_file.exists():
            error_file.unlink()


    except Exception as error:

        status_record["status"] = "failed"

        status_record["error"] = (
            f"{type(error).__name__}: {error}"
        )

        error_file.write_text(
            status_record["error"],
            encoding="utf-8",
        )


    download_log.append(
        status_record
    )


    # Continuously save progress so the workflow can resume.
    pd.DataFrame(
        download_log
    ).to_csv(
        TABLE_DIR / "download_log.csv",
        index=False,
    )


# ============================================================
# 11. REVIEW DOWNLOAD STATUS
# ============================================================

download_log_df = pd.DataFrame(
    download_log
)

status_summary = (
    download_log_df["status"]
    .value_counts(
        dropna=False
    )
    .rename_axis("status")
    .reset_index(
        name="stations"
    )
)

print("\nDownload status summary")

display(
    status_summary
)


failed_downloads = download_log_df.loc[
    download_log_df["status"] == "failed"
].copy()


if not failed_downloads.empty:

    print(
        "\nThe following stations failed. "
        "Rerun this cell to retry them. "
        "Completed files will be reused."
    )

    display(
        failed_downloads
    )


# ============================================================
# 12. COMBINE ALL SUCCESSFUL RESULTS
# ============================================================

if not all_station_tables:

    raise RuntimeError(
        "No NOAA station responses were parsed successfully. "
        "Review the download log and error files."
    )


pds_long = pd.concat(
    all_station_tables,
    ignore_index=True,
)


if (
    DATA_TYPE == "depth"
    and UNITS == "metric"
):

    value_column_name = (
        "precipitation_depth_mm"
    )

elif (
    DATA_TYPE == "depth"
    and UNITS == "english"
):

    value_column_name = (
        "precipitation_depth_inches"
    )

else:

    value_column_name = (
        "precipitation_value"
    )


pds_long = pds_long.rename(
    columns={
        "precipitation_value": (
            value_column_name
        )
    }
)


pds_long = (
    pds_long
    .sort_values(
        [
            "station_id",
            "estimate_type",
            "duration",
            "ari_years",
        ]
    )
    .reset_index(drop=True)
)


long_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_estimates_long.csv"
)

pds_long.to_csv(
    long_output,
    index=False,
)


print("\nLong-format results")

display(
    pds_long.head(20)
)


# ============================================================
# 13. CREATE A WIDE TABLE
# ============================================================

wide_index_columns = [
    "station_id",
    "station_name",
    "state",
    "latitude",
    "longitude",
    "estimate_type",
    "duration",
]


pds_wide = (
    pds_long
    .pivot_table(
        index=wide_index_columns,
        columns="ari_years",
        values=value_column_name,
        aggfunc="first",
    )
    .reset_index()
)


renamed_columns = []

for column in pds_wide.columns:

    if isinstance(
        column,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        numeric_column = float(
            column
        )

        if numeric_column.is_integer():

            renamed_columns.append(
                f"ari_{int(numeric_column)}yr"
            )

        else:

            renamed_columns.append(
                f"ari_{numeric_column:g}yr"
            )

    else:

        renamed_columns.append(
            str(column)
        )


pds_wide.columns = renamed_columns


wide_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_estimates_wide.csv"
)

pds_wide.to_csv(
    wide_output,
    index=False,
)


print("\nWide-format results")

display(
    pds_wide.head(20)
)


# ============================================================
# 14. CREATE QUALITY-CONTROL SUMMARY
# ============================================================

successful_statuses = [
    "downloaded",
    "reused_existing",
]


successful_station_count = (
    download_log_df.loc[
        download_log_df["status"].isin(
            successful_statuses
        ),
        "station_id",
    ]
    .nunique()
)


qc_summary = pd.DataFrame(
    {
        "metric": [
            "valid workbook rows",
            "unique stations requested",
            "successful stations",
            "failed stations",
            "long-table rows",
            "unique durations",
            "unique recurrence intervals",
            "estimate types",
            "requested units",
        ],
        "value": [
            len(station_rows),
            len(stations),
            successful_station_count,
            len(failed_downloads),
            len(pds_long),
            pds_long["duration"].nunique(),
            pds_long["ari_years"].nunique(),
            ", ".join(
                sorted(
                    pds_long[
                        "estimate_type"
                    ].unique()
                )
            ),
            UNITS,
        ],
    }
)


qc_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_qc_summary.csv"
)

qc_summary.to_csv(
    qc_output,
    index=False,
)


print("\nQuality-control summary")

display(
    qc_summary
)


# ============================================================
# 15. FINAL OUTPUT LOCATIONS
# ============================================================

print("\n" + "=" * 65)
print("NOAA ATLAS 14 DOWNLOAD PIPELINE COMPLETE")
print("=" * 65)

print("\nStation inventory:")
print(station_inventory_path)

print("\nRaw NOAA station responses:")
print(RAW_DIR)

print("\nLong-format combined table:")
print(long_output)

print("\nWide-format combined table:")
print(wide_output)

print("\nDownload log:")
print(TABLE_DIR / "download_log.csv")

print("\nQuality-control summary:")
print(qc_output)

print("\nFailed-station error files:")
print(ERROR_DIR)

C:\Users\ioolajide\AppData\Local\anaconda3\envs\stage4\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Current notebook folder:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14

Station file:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\al.xls

Output folder:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS

Columns found in al.xls:
 - State
 - Station name
 - Station ID
 - Post-merge station ID
 - Co-located station ID
 - Base duration
 - Source of data
 - Latitude
 - Longitude
 - Elevation (ft)
 - Period of record


C:\Users\ioolajide\AppData\Local\Temp\2\ipykernel_10192\2541161631.py:416: UserWarning: 2 station IDs have more than one coordinate. The first coordinate will be used.
  warnings.warn(



Station inventory summary
-------------------------
Workbook records: 283
Unique stations: 225


,Station ID,State,Station name,Latitude,Longitude,Post-merge station ID,Co-located station ID,Base duration,Source of data,Period of record,Elevation (ft)
0,01-0008,AL,ABBEVILLE,31.5703,-85.2483,,,1-day; 1-hour,NCDC,7/1948-8/2010; 6/1948-12/2010,456
1,01-0063,AL,ADDISON,34.2031,-87.1814,,,1-day; 1-hour; 15-min,NCDC,3/1938-10/2011; 6/1948-12/2010; 10/1976-12/2010,766
2,01-0140,AL,ALBERTA,32.2322,-87.4106,,,1-day; 1-hour; 15-min,NCDC,10/1940-10/2011; 9/1963-12/2010; 5/1971-12/2010,175
3,01-0148,AL,ALBERTVILLE 2 SE,34.2333,-86.1667,01-0957,,1-day,NCDC,1/1908-3/1977,1142
4,01-0178,AL,ALICEVILLE,33.1272,-88.1550,,,1-day,NCDC,3/1934-7/2011,195
5,01-0184,AL,ALICEVILLE L&D,33.2100,-88.2878,,,1-day,NCDC,2/1940-8/2010,165
6,01-0252,AL,ANDALUSIA 3 W,31.3067,-86.5222,,,1-day; 1-hour; 15-min,NCDC,10/1912-8/2011; 6/1948-12/2010; 3/1980-12/2010,250
7,01-0272,AL,ANNISTON METRO AP,33.5872,-85.8556,,,1-day,NCDC,3/1903-10/2010,594
8,01-0338,AL,ARLEY 1 S,34.0667,-87.2333,,,1-day,NCDC,3/1938-10/1983,745
9,01-0369,AL,ASHLAND 3 ENE,33.2942,-85.7789,,,1-day; 1-hour; 15-min,NCDC,3/1940-10/2011; 6/1948-12/2010; 8/1972-12/2010,1022



Download status summary


,status,stations
0,failed,225



The following stations failed. Rerun this cell to retry them. Completed files will be reused.


,station_id,station_name,latitude,longitude,raw_file,status,request_url,rows_parsed,error
0,01-0008,ABBEVILLE,31.5703,-85.2483,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
1,01-0063,ADDISON,34.2031,-87.1814,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
2,01-0140,ALBERTA,32.2322,-87.4106,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
3,01-0148,ALBERTVILLE 2 SE,34.2333,-86.1667,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
4,01-0178,ALICEVILLE,33.1272,-88.1550,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
...,...,...,...,...,...,...,...,...,...
220,01-8867,WHATLEY,31.6508,-87.7097,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
221,01-8925,WHITFIELD LOCK 3,32.2833,-88.0167,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
222,01-8998,WINFIELD 2 SW,33.9111,-87.8475,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."
223,01-9103,YATES HYDRO PLT,32.5667,-85.9000,D:\projstore\Ismail\Dr Koriche\Intense Rainfal...,failed,https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text....,0,"ValueError: The NOAA response was recognized, ..."


RuntimeError: No NOAA station responses were parsed successfully. Review the download log and error files.

In [2]:
# ============================================================
# INSPECT STATIONS WITH CONFLICTING COORDINATES
# ============================================================

conflicting_ids = coordinate_conflicts["Station ID"].tolist()

conflicting_station_records = (
    station_rows.loc[
        station_rows["Station ID"].isin(conflicting_ids)
    ]
    .sort_values(
        [
            "Station ID",
            "Latitude",
            "Longitude",
        ]
    )
    .reset_index(drop=True)
)

display(
    conflicting_station_records[
        [
            column
            for column in [
                "Station ID",
                "Station name",
                "Latitude",
                "Longitude",
                "Elevation (ft)",
                "Base duration",
                "Post-merge station ID",
                "Co-located station ID",
                "Period of record",
                "Source of data",
            ]
            if column in conflicting_station_records.columns
        ]
    ]
)

,Station ID,Station name,Latitude,Longitude,Elevation (ft),Base duration,Post-merge station ID,Co-located station ID,Period of record,Source of data
0,01-2172,DAUPHIN ISLAND #2,30.2500,-88.0833,8,1-day,,NaN,6/1866-10/2011,NCDC
1,01-2172,DAUPHIN ISLAND #2,30.2500,-88.0833,8,15-min,,NaN,8/1975-12/2010,NCDC
2,01-2172,DAUPHIN ISLAND #2,30.2505,-88.0775,8,1-hour,,NaN,6/1948-12/2010,NCDC
3,01-3519,GREENVILLE,31.7901,-86.6087,342,1-day,,NaN,9/1900-10/2011,NCDC
4,01-3519,GREENVILLE,31.7901,-86.6087,342,1-hour,,NaN,6/1948-12/2010,NCDC
5,01-3519,GREENVILLE,31.7947,-86.6147,342,15-min,,NaN,9/1971-12/2010,NCDC


In [3]:
# ============================================================
# MEASURE DISTANCE BETWEEN CONFLICTING COORDINATES
# ============================================================

from math import radians, sin, cos, asin, sqrt


def haversine_km(
    latitude_1,
    longitude_1,
    latitude_2,
    longitude_2,
):
    """
    Calculate great-circle distance between two coordinates.
    """

    earth_radius_km = 6371.0088

    lat1 = radians(latitude_1)
    lon1 = radians(longitude_1)
    lat2 = radians(latitude_2)
    lon2 = radians(longitude_2)

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = (
        sin(delta_lat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(delta_lon / 2) ** 2
    )

    return 2 * earth_radius_km * asin(sqrt(a))


distance_rows = []

for station_id, group in conflicting_station_records.groupby(
    "Station ID"
):
    unique_coordinates = (
        group[
            [
                "Latitude",
                "Longitude",
            ]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    first_coordinate = unique_coordinates.iloc[0]

    for coordinate_index in range(
        1,
        len(unique_coordinates),
    ):
        other_coordinate = unique_coordinates.iloc[
            coordinate_index
        ]

        distance_rows.append(
            {
                "Station ID": station_id,
                "Latitude 1": first_coordinate["Latitude"],
                "Longitude 1": first_coordinate["Longitude"],
                "Latitude 2": other_coordinate["Latitude"],
                "Longitude 2": other_coordinate["Longitude"],
                "Distance km": haversine_km(
                    first_coordinate["Latitude"],
                    first_coordinate["Longitude"],
                    other_coordinate["Latitude"],
                    other_coordinate["Longitude"],
                ),
            }
        )

coordinate_distance_table = pd.DataFrame(
    distance_rows
)

display(coordinate_distance_table)

,Station ID,Latitude 1,Longitude 1,Latitude 2,Longitude 2,Distance km
0,01-2172,30.2500,-88.0833,30.2505,-88.0775,0.559881
1,01-3519,31.7901,-86.6087,31.7947,-86.6147,0.763674


In [4]:
# ============================================================
# DIAGNOSE THE NOAA ATLAS 14 DOWNLOAD FAILURE
# ============================================================

from pathlib import Path

import pandas as pd
import requests

from IPython.display import display


# ------------------------------------------------------------
# 1. Output folders used by the previous code
# ------------------------------------------------------------

OUTPUT_ROOT = Path.cwd() / "NOAA_Atlas14_Alabama_PDS"
TABLE_DIR = OUTPUT_ROOT / "tables"
RAW_DIR = OUTPUT_ROOT / "raw_noaa_responses"
ERROR_DIR = OUTPUT_ROOT / "errors"

DOWNLOAD_LOG = TABLE_DIR / "download_log.csv"
STATION_FILE = Path.cwd() / "al.xls"

print("Notebook folder:")
print(Path.cwd())

print("\nDownload log:")
print(DOWNLOAD_LOG)

print("\nRaw response folder:")
print(RAW_DIR)

print("\nError folder:")
print(ERROR_DIR)


# ------------------------------------------------------------
# 2. Inspect the download log
# ------------------------------------------------------------

if DOWNLOAD_LOG.exists():

    download_log = pd.read_csv(DOWNLOAD_LOG)

    print("\nDownload status counts:")
    display(
        download_log["status"]
        .value_counts(dropna=False)
        .rename_axis("status")
        .reset_index(name="number_of_stations")
    )

    if "error" in download_log.columns:

        distinct_errors = (
            download_log.loc[
                download_log["error"].notna(),
                [
                    "station_id",
                    "station_name",
                    "error",
                ],
            ]
            .drop_duplicates(subset=["error"])
            .head(20)
        )

        print("\nFirst distinct errors:")
        display(distinct_errors)

else:

    print("\nThe previous download log was not found.")


# ------------------------------------------------------------
# 3. Inspect saved error files
# ------------------------------------------------------------

error_files = sorted(
    ERROR_DIR.glob("*.error.txt")
)

print(
    f"\nNumber of saved error files: "
    f"{len(error_files):,}"
)

for error_file in error_files[:3]:

    print("\n" + "=" * 70)
    print("ERROR FILE:", error_file.name)
    print("=" * 70)

    error_text = error_file.read_text(
        encoding="utf-8",
        errors="replace",
    )

    print(error_text[:2000])


# ------------------------------------------------------------
# 4. Inspect any raw NOAA response files
# ------------------------------------------------------------

raw_files = sorted(
    RAW_DIR.glob("*.csv")
)

print(
    f"\nNumber of raw NOAA response files: "
    f"{len(raw_files):,}"
)

if raw_files:

    first_raw_file = raw_files[0]

    first_raw_text = first_raw_file.read_text(
        encoding="utf-8",
        errors="replace",
    )

    print("\n" + "=" * 70)
    print("FIRST RAW FILE:", first_raw_file.name)
    print("=" * 70)

    print(first_raw_text[:3000])

else:

    print(
        "\nNo raw NOAA responses were saved. "
        "The requests probably failed before the save step."
    )


# ------------------------------------------------------------
# 5. Independently test one Alabama station
# ------------------------------------------------------------

if not STATION_FILE.exists():

    raise FileNotFoundError(
        f"Could not find the station workbook:\n{STATION_FILE}"
    )


stations_test = pd.read_excel(
    STATION_FILE,
    engine="xlrd",
)

first_station = stations_test.dropna(
    subset=[
        "Latitude",
        "Longitude",
    ]
).iloc[0]

station_id = str(
    first_station["Station ID"]
).strip()

station_name = str(
    first_station["Station name"]
).strip()

latitude = float(
    first_station["Latitude"]
)

longitude = float(
    first_station["Longitude"]
)


NOAA_API_URL = (
    "https://hdsc.nws.noaa.gov/cgi-bin/new/fe_text.csv"
)

parameters = {
    "lat": f"{latitude:.6f}",
    "lon": f"{longitude:.6f}",
    "data": "depth",
    "units": "metric",
    "series": "pds",
}

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/150.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/csv,text/plain,text/html,"
        "application/xhtml+xml,*/*"
    ),
    "Referer": (
        "https://hdsc.nws.noaa.gov/"
        "pfds/pfds_map_cont.html?bkmrk=al"
    ),
}


print("\n" + "=" * 70)
print("TESTING ONE STATION")
print("=" * 70)

print("Station ID:", station_id)
print("Station name:", station_name)
print("Latitude:", latitude)
print("Longitude:", longitude)


try:

    response = requests.get(
        NOAA_API_URL,
        params=parameters,
        headers=headers,
        timeout=120,
    )

    print("\nHTTP status:")
    print(response.status_code)

    print("\nFinal request URL:")
    print(response.url)

    print("\nContent type:")
    print(response.headers.get("Content-Type"))

    print("\nResponse size:")
    print(f"{len(response.content):,} bytes")

    print("\nResponse preview:")
    print("-" * 70)
    print(response.text[:4000])
    print("-" * 70)

    response.raise_for_status()

except Exception as error:

    print("\nREQUEST ERROR:")
    print(
        f"{type(error).__name__}: {error}"
    )

Notebook folder:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14

Download log:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\download_log.csv

Raw response folder:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\raw_noaa_responses

Error folder:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\errors

Download status counts:


,status,number_of_stations
0,failed,225



First distinct errors:


,station_id,station_name,error
0,01-0008,ABBEVILLE,"ValueError: The NOAA response was recognized, ..."



Number of saved error files: 225

ERROR FILE: 01-0008.error.txt
ValueError: The NOAA response was recognized, but no duration and recurrence-interval records were parsed.

ERROR FILE: 01-0063.error.txt
ValueError: The NOAA response was recognized, but no duration and recurrence-interval records were parsed.

ERROR FILE: 01-0140.error.txt
ValueError: The NOAA response was recognized, but no duration and recurrence-interval records were parsed.

Number of raw NOAA response files: 225

FIRST RAW FILE: 01-0008_pds_frequency.csv
Point precipitation frequency estimates (millimeters)
NOAA Atlas 14 Volume 9 Version 2
Data type: Precipitation depth
Time series type: Partial duration
Project area: Southeastern States
Location name (ESRI Maps): None
Station Name: None
Latitude: 31.570300 Degree
Longitude: -85.248300 Degree
Elevation (USGS): None None


PRECIPITATION FREQUENCY ESTIMATES
by duration for ARI (years):, 1,2,5,10,25,50,100,200,500,1000
5-min:, 13,15,18,21,24,26,29,31,34,37
10-min:, 19

In [5]:
# ============================================================
# REPAIR NOAA ATLAS 14 PDS PARSING
# REUSE ALL 225 RAW FILES ALREADY DOWNLOADED
# ============================================================

from __future__ import annotations

import csv
import io
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from IPython.display import display
from tqdm.auto import tqdm


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path.cwd()

OUTPUT_ROOT = (
    BASE_DIR
    / "NOAA_Atlas14_Alabama_PDS"
)

RAW_DIR = (
    OUTPUT_ROOT
    / "raw_noaa_responses"
)

TABLE_DIR = (
    OUTPUT_ROOT
    / "tables"
)

ERROR_DIR = (
    OUTPUT_ROOT
    / "errors"
)

STATION_INVENTORY_FILE = (
    TABLE_DIR
    / "alabama_unique_station_inventory.csv"
)

ORIGINAL_DOWNLOAD_LOG = (
    TABLE_DIR
    / "download_log.csv"
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ERROR_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("Raw NOAA response folder:")
print(RAW_DIR)

print("\nStation inventory:")
print(STATION_INVENTORY_FILE)


if not RAW_DIR.exists():

    raise FileNotFoundError(
        f"Raw NOAA response folder not found:\n{RAW_DIR}"
    )


if not STATION_INVENTORY_FILE.exists():

    raise FileNotFoundError(
        "The station inventory was not found:\n"
        f"{STATION_INVENTORY_FILE}"
    )


# ============================================================
# 2. LOAD UNIQUE ALABAMA STATION INVENTORY
# ============================================================

stations = pd.read_csv(
    STATION_INVENTORY_FILE,
    dtype={
        "Station ID": str,
        "Station name": str,
        "State": str,
        "Post-merge station ID": str,
        "Co-located station ID": str,
        "Base duration": str,
        "Source of data": str,
        "Period of record": str,
    },
)


stations["Station ID"] = (
    stations["Station ID"]
    .astype(str)
    .str.strip()
)


stations["Station name"] = (
    stations["Station name"]
    .astype(str)
    .str.strip()
)


stations["Latitude"] = pd.to_numeric(
    stations["Latitude"],
    errors="coerce",
)


stations["Longitude"] = pd.to_numeric(
    stations["Longitude"],
    errors="coerce",
)


station_lookup = (
    stations
    .drop_duplicates(
        subset=["Station ID"]
    )
    .set_index("Station ID")
)


print(
    f"\nUnique stations in inventory: "
    f"{len(station_lookup):,}"
)


# ============================================================
# 3. NOAA RESPONSE VALIDATION
# ============================================================

def is_valid_noaa_response(
    text: str,
) -> bool:
    """
    Confirm that a raw response contains all three NOAA
    precipitation-frequency tables.
    """

    upper_text = text.upper()

    required_phrases = [
        "PRECIPITATION FREQUENCY ESTIMATES",
        "BY DURATION FOR ARI",
        "UPPER BOUND OF 90% CONFIDENCE INTERVAL",
        "LOWER BOUND OF 90% CONFIDENCE INTERVAL",
    ]

    return all(
        phrase in upper_text
        for phrase in required_phrases
    )


# NOAA duration labels include:
# 5-min, 60-min, 2-hr, 24-hr, 2-day, 60-day
DURATION_PATTERN = re.compile(
    r"^\d+(?:\.\d+)?-"
    r"(?:min|mins|minute|minutes|"
    r"hr|hrs|hour|hours|"
    r"day|days)$",
    flags=re.IGNORECASE,
)


# ============================================================
# 4. NUMERIC AND DURATION HELPERS
# ============================================================

def parse_number(
    value: Any,
) -> float:
    """
    Convert a NOAA CSV field to a floating-point value.
    """

    if value is None:
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    text = text.replace(
        ",",
        "",
    )

    if text.upper() in {
        "NA",
        "N/A",
        "NAN",
        "NONE",
        "-",
        "--",
    }:
        return np.nan

    text = text.strip(
        "()[]{}"
    )

    try:
        return float(text)

    except ValueError:
        return np.nan


def normalize_duration(
    duration: str,
) -> str:
    """
    Standardize NOAA duration labels.
    """

    duration = (
        str(duration)
        .strip()
        .rstrip(":")
        .lower()
    )

    replacements = {
        "mins": "min",
        "minute": "min",
        "minutes": "min",
        "hrs": "hr",
        "hour": "hr",
        "hours": "hr",
        "days": "day",
    }

    for old, new in replacements.items():

        if duration.endswith(
            f"-{old}"
        ):

            duration = (
                duration[
                    : -len(old)
                ]
                + new
            )

    return duration


def duration_to_hours(
    duration: str,
) -> float:
    """
    Convert a standardized NOAA duration to hours.
    """

    duration = normalize_duration(
        duration
    )

    number_text, unit = duration.split(
        "-",
        maxsplit=1,
    )

    number = float(
        number_text
    )

    if unit == "min":
        return number / 60.0

    if unit == "hr":
        return number

    if unit == "day":
        return number * 24.0

    return np.nan


def parse_ari_header(
    cells: list[str],
) -> list[float]:
    """
    Extract recurrence intervals from the separate NOAA
    'by duration for ARI' row.
    """

    ari_values = []

    for cell in cells[1:]:

        value = parse_number(
            cell
        )

        if np.isfinite(value):

            ari_values.append(
                float(value)
            )

    return ari_values


# ============================================================
# 5. EXTRACT OPTIONAL METADATA FROM NOAA RESPONSE
# ============================================================

def extract_response_metadata(
    text: str,
) -> dict[str, Any]:
    """
    Extract general metadata printed above the NOAA tables.
    """

    metadata = {}

    field_patterns = {
        "noaa_product": (
            r"^Point precipitation frequency estimates "
            r"\((.*?)\)"
        ),
        "atlas_version": (
            r"^(NOAA Atlas .*?)$"
        ),
        "noaa_data_type": (
            r"^Data type:\s*(.*?)$"
        ),
        "time_series_type": (
            r"^Time series type:\s*(.*?)$"
        ),
        "project_area": (
            r"^Project area:\s*(.*?)$"
        ),
        "noaa_station_name": (
            r"^Station Name:\s*(.*?)$"
        ),
        "response_latitude": (
            r"^Latitude:\s*"
            r"([-+]?\d+(?:\.\d+)?)"
        ),
        "response_longitude": (
            r"^Longitude:\s*"
            r"([-+]?\d+(?:\.\d+)?)"
        ),
    }

    lines = text.splitlines()

    for output_name, pattern in field_patterns.items():

        for line in lines:

            match = re.match(
                pattern,
                line.strip(),
                flags=re.IGNORECASE,
            )

            if match:

                value = match.group(1).strip()

                if output_name in {
                    "response_latitude",
                    "response_longitude",
                }:

                    value = parse_number(
                        value
                    )

                metadata[output_name] = value
                break

    return metadata


# ============================================================
# 6. CORRECTED NOAA PARSER
# ============================================================

def parse_noaa_frequency_response(
    text: str,
    station_metadata: dict[str, Any],
) -> pd.DataFrame:
    """
    Parse one NOAA Atlas 14 PFDS response.

    The recurrence-interval header occurs on the line following
    each precipitation-frequency section title. This function
    handles those lines separately.
    """

    if not is_valid_noaa_response(
        text
    ):

        preview = (
            text[:500]
            .replace("\n", " | ")
        )

        raise ValueError(
            "The file is not a recognized NOAA Atlas 14 "
            f"frequency response. Preview: {preview}"
        )


    response_metadata = extract_response_metadata(
        text
    )


    reader = csv.reader(
        io.StringIO(
            text.lstrip("\ufeff")
        )
    )


    current_estimate_type = None
    current_ari_values = []

    output_rows = []


    for raw_row in reader:

        cells = [
            str(cell).strip()
            for cell in raw_row
        ]


        if not cells:
            continue


        first_cell = (
            cells[0]
            .strip()
        )


        joined_upper = (
            " ".join(cells)
            .upper()
        )


        # ----------------------------------------------------
        # Identify the main estimate section
        # ----------------------------------------------------

        if (
            "PRECIPITATION FREQUENCY ESTIMATES"
            in joined_upper
            and "UPPER BOUND"
            not in joined_upper
            and "LOWER BOUND"
            not in joined_upper
        ):

            current_estimate_type = (
                "estimate"
            )

            current_ari_values = []

            continue


        # ----------------------------------------------------
        # Identify the upper 90% confidence section
        # ----------------------------------------------------

        if (
            "PRECIPITATION FREQUENCY ESTIMATES"
            in joined_upper
            and "UPPER BOUND"
            in joined_upper
        ):

            current_estimate_type = (
                "upper_90"
            )

            current_ari_values = []

            continue


        # ----------------------------------------------------
        # Identify the lower 90% confidence section
        # ----------------------------------------------------

        if (
            "PRECIPITATION FREQUENCY ESTIMATES"
            in joined_upper
            and "LOWER BOUND"
            in joined_upper
        ):

            current_estimate_type = (
                "lower_90"
            )

            current_ari_values = []

            continue


        # ----------------------------------------------------
        # Read the recurrence intervals from their own line
        # ----------------------------------------------------

        if (
            current_estimate_type is not None
            and "BY DURATION FOR ARI"
            in joined_upper
        ):

            current_ari_values = (
                parse_ari_header(cells)
            )


            if not current_ari_values:

                raise ValueError(
                    "A frequency-table section was found, "
                    "but its recurrence-interval header "
                    "contained no numeric ARIs."
                )

            continue


        # Ignore all rows before a section and ARI header.
        if (
            current_estimate_type is None
            or not current_ari_values
        ):

            continue


        duration = normalize_duration(
            first_cell
        )


        if not DURATION_PATTERN.match(
            duration
        ):

            continue


        expected_value_count = len(
            current_ari_values
        )


        value_cells = cells[
            1:
            1 + expected_value_count
        ]


        values = [
            parse_number(cell)
            for cell in value_cells
        ]


        if len(values) != expected_value_count:

            raise ValueError(
                f"Duration {duration!r} has "
                f"{len(values)} values, but "
                f"{expected_value_count} ARIs were expected."
            )


        duration_hours = duration_to_hours(
            duration
        )


        for ari_years, precipitation_mm in zip(
            current_ari_values,
            values,
        ):

            output_row = dict(
                station_metadata
            )

            output_row.update(
                response_metadata
            )

            output_row.update(
                {
                    "series": "pds",
                    "estimate_type": (
                        current_estimate_type
                    ),
                    "duration": duration,
                    "duration_hours": (
                        duration_hours
                    ),
                    "ari_years": ari_years,
                    "precipitation_depth_mm": (
                        precipitation_mm
                    ),
                }
            )

            output_rows.append(
                output_row
            )


    parsed = pd.DataFrame(
        output_rows
    )


    if parsed.empty:

        raise ValueError(
            "The NOAA response was recognized, but no "
            "frequency-estimate rows were extracted."
        )


    expected_types = {
        "estimate",
        "upper_90",
        "lower_90",
    }


    found_types = set(
        parsed["estimate_type"]
        .dropna()
        .unique()
    )


    if found_types != expected_types:

        raise ValueError(
            "The response did not produce all expected "
            "estimate types. Expected "
            f"{sorted(expected_types)}, found "
            f"{sorted(found_types)}."
        )


    expected_aris = {
        1.0,
        2.0,
        5.0,
        10.0,
        25.0,
        50.0,
        100.0,
        200.0,
        500.0,
        1000.0,
    }


    found_aris = set(
        parsed["ari_years"]
        .dropna()
        .unique()
    )


    if found_aris != expected_aris:

        warnings.warn(
            "Parsed recurrence intervals differ from the "
            "standard NOAA set. Found: "
            f"{sorted(found_aris)}"
        )


    return parsed


# ============================================================
# 7. TEST THE CORRECTED PARSER ON ONE RAW FILE
# ============================================================

raw_files = sorted(
    RAW_DIR.glob(
        "*_pds_frequency.csv"
    )
)


if not raw_files:

    raise FileNotFoundError(
        f"No raw NOAA response files were found in:\n{RAW_DIR}"
    )


print(
    f"\nRaw NOAA files found: "
    f"{len(raw_files):,}"
)


first_file = raw_files[0]


first_station_id = (
    first_file.stem
    .replace(
        "_pds_frequency",
        "",
    )
)


if first_station_id not in station_lookup.index:

    raise KeyError(
        f"Station {first_station_id!r} from "
        f"{first_file.name} was not found in the inventory."
    )


first_station = station_lookup.loc[
    first_station_id
]


first_metadata = {
    "station_id": first_station_id,
    "station_name": (
        first_station.get(
            "Station name",
            np.nan,
        )
    ),
    "state": (
        first_station.get(
            "State",
            "AL",
        )
    ),
    "latitude": (
        first_station.get(
            "Latitude",
            np.nan,
        )
    ),
    "longitude": (
        first_station.get(
            "Longitude",
            np.nan,
        )
    ),
}


first_text = first_file.read_text(
    encoding="utf-8",
    errors="replace",
)


first_parsed = (
    parse_noaa_frequency_response(
        first_text,
        first_metadata,
    )
)


print(
    "\nCorrected parser test succeeded."
)

print(
    f"Rows parsed from {first_file.name}: "
    f"{len(first_parsed):,}"
)

print(
    "Estimate types:",
    sorted(
        first_parsed[
            "estimate_type"
        ].unique()
    ),
)

print(
    "Durations:",
    first_parsed[
        "duration"
    ].nunique(),
)

print(
    "Recurrence intervals:",
    first_parsed[
        "ari_years"
    ].nunique(),
)


display(
    first_parsed.head(20)
)


# ============================================================
# 8. PARSE ALL 225 EXISTING RAW FILES
# ============================================================

all_station_tables = []

reparse_log = []


for raw_file in tqdm(
    raw_files,
    desc="Reparsing existing NOAA files",
):

    station_id = (
        raw_file.stem
        .replace(
            "_pds_frequency",
            "",
        )
    )


    status_record = {
        "station_id": station_id,
        "raw_file": str(raw_file),
        "status": None,
        "rows_parsed": 0,
        "error": None,
    }


    try:

        if station_id not in station_lookup.index:

            raise KeyError(
                f"Station ID {station_id!r} was not found "
                "in the station inventory."
            )


        station = station_lookup.loc[
            station_id
        ]


        metadata = {
            "station_id": station_id,
            "station_name": station.get(
                "Station name",
                np.nan,
            ),
            "state": station.get(
                "State",
                "AL",
            ),
            "latitude": station.get(
                "Latitude",
                np.nan,
            ),
            "longitude": station.get(
                "Longitude",
                np.nan,
            ),
        }


        optional_metadata = {
            "post_merge_station_id": (
                "Post-merge station ID"
            ),
            "co_located_station_id": (
                "Co-located station ID"
            ),
            "base_duration": (
                "Base duration"
            ),
            "source_of_data": (
                "Source of data"
            ),
            "elevation_ft": (
                "Elevation (ft)"
            ),
            "period_of_record": (
                "Period of record"
            ),
        }


        for output_name, input_name in (
            optional_metadata.items()
        ):

            if input_name in station.index:

                metadata[output_name] = (
                    station.get(
                        input_name,
                        np.nan,
                    )
                )


        text = raw_file.read_text(
            encoding="utf-8",
            errors="replace",
        )


        parsed_station = (
            parse_noaa_frequency_response(
                text,
                metadata,
            )
        )


        all_station_tables.append(
            parsed_station
        )


        status_record["status"] = (
            "reparsed_successfully"
        )

        status_record["rows_parsed"] = len(
            parsed_station
        )


        old_error_file = (
            ERROR_DIR
            / f"{station_id}.error.txt"
        )


        if old_error_file.exists():

            old_error_file.unlink()


    except Exception as error:

        status_record["status"] = (
            "reparse_failed"
        )

        status_record["error"] = (
            f"{type(error).__name__}: {error}"
        )


        new_error_file = (
            ERROR_DIR
            / f"{station_id}.reparse_error.txt"
        )


        new_error_file.write_text(
            status_record["error"],
            encoding="utf-8",
        )


    reparse_log.append(
        status_record
    )


    # Save a continuous checkpoint.
    pd.DataFrame(
        reparse_log
    ).to_csv(
        TABLE_DIR
        / "reparse_log.csv",
        index=False,
    )


reparse_log_df = pd.DataFrame(
    reparse_log
)


print("\nReparse status summary:")


display(
    reparse_log_df[
        "status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "status"
    )
    .reset_index(
        name="stations"
    )
)


failed_reparses = (
    reparse_log_df.loc[
        reparse_log_df[
            "status"
        ] == "reparse_failed"
    ]
    .copy()
)


if not failed_reparses.empty:

    print("\nFiles that still failed:")

    display(
        failed_reparses
    )


if not all_station_tables:

    raise RuntimeError(
        "No raw NOAA files were parsed successfully."
    )


# ============================================================
# 9. COMBINE ALL SUCCESSFUL STATIONS
# ============================================================

pds_long = pd.concat(
    all_station_tables,
    ignore_index=True,
)


pds_long = (
    pds_long
    .sort_values(
        [
            "station_id",
            "estimate_type",
            "duration_hours",
            "ari_years",
        ]
    )
    .reset_index(
        drop=True
    )
)


long_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_estimates_long.csv"
)


pds_long.to_csv(
    long_output,
    index=False,
)


print("\nCombined long-format table:")

print(
    f"Rows: {len(pds_long):,}"
)

print(
    "Stations:",
    pds_long[
        "station_id"
    ].nunique(),
)

print(
    "Durations:",
    pds_long[
        "duration"
    ].nunique(),
)

print(
    "ARIs:",
    pds_long[
        "ari_years"
    ].nunique(),
)


display(
    pds_long.head(30)
)


# ============================================================
# 10. CREATE A WIDE TABLE
# ============================================================

wide_index_columns = [
    "station_id",
    "station_name",
    "state",
    "latitude",
    "longitude",
    "estimate_type",
    "duration",
    "duration_hours",
]


pds_wide = (
    pds_long
    .pivot_table(
        index=wide_index_columns,
        columns="ari_years",
        values="precipitation_depth_mm",
        aggfunc="first",
    )
    .reset_index()
)


renamed_columns = []


for column in pds_wide.columns:

    if isinstance(
        column,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        numeric_column = float(
            column
        )


        if numeric_column.is_integer():

            renamed_columns.append(
                f"ari_{int(numeric_column)}yr_mm"
            )

        else:

            renamed_columns.append(
                f"ari_{numeric_column:g}yr_mm"
            )

    else:

        renamed_columns.append(
            str(column)
        )


pds_wide.columns = (
    renamed_columns
)


pds_wide = (
    pds_wide
    .sort_values(
        [
            "station_id",
            "estimate_type",
            "duration_hours",
        ]
    )
    .reset_index(
        drop=True
    )
)


wide_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_estimates_wide.csv"
)


pds_wide.to_csv(
    wide_output,
    index=False,
)


print("\nCombined wide-format table:")

display(
    pds_wide.head(30)
)


# ============================================================
# 11. CREATE A TABLE WITH ESTIMATE AND CONFIDENCE BOUNDS
# ============================================================

confidence_table = (
    pds_long
    .pivot_table(
        index=[
            "station_id",
            "station_name",
            "state",
            "latitude",
            "longitude",
            "duration",
            "duration_hours",
            "ari_years",
        ],
        columns="estimate_type",
        values="precipitation_depth_mm",
        aggfunc="first",
    )
    .reset_index()
)


confidence_table.columns.name = None


confidence_table = confidence_table.rename(
    columns={
        "estimate": (
            "estimate_mm"
        ),
        "lower_90": (
            "lower_90_mm"
        ),
        "upper_90": (
            "upper_90_mm"
        ),
    }
)


confidence_table["confidence_width_mm"] = (
    confidence_table["upper_90_mm"]
    - confidence_table["lower_90_mm"]
)


confidence_table["lower_difference_mm"] = (
    confidence_table["estimate_mm"]
    - confidence_table["lower_90_mm"]
)


confidence_table["upper_difference_mm"] = (
    confidence_table["upper_90_mm"]
    - confidence_table["estimate_mm"]
)


confidence_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_estimates_with_confidence_bounds.csv"
)


confidence_table.to_csv(
    confidence_output,
    index=False,
)


print(
    "\nEstimate and confidence-bound table:"
)

display(
    confidence_table.head(30)
)


# ============================================================
# 12. QUALITY-CONTROL CHECKS
# ============================================================

station_count = (
    pds_long[
        "station_id"
    ].nunique()
)


duration_count = (
    pds_long[
        "duration"
    ].nunique()
)


ari_count = (
    pds_long[
        "ari_years"
    ].nunique()
)


estimate_type_count = (
    pds_long[
        "estimate_type"
    ].nunique()
)


expected_rows_per_station = (
    duration_count
    * ari_count
    * estimate_type_count
)


station_row_counts = (
    pds_long
    .groupby(
        [
            "station_id",
            "station_name",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "parsed_rows"
        }
    )
)


station_row_counts[
    "expected_rows"
] = expected_rows_per_station


station_row_counts[
    "complete"
] = (
    station_row_counts[
        "parsed_rows"
    ]
    == expected_rows_per_station
)


station_row_counts.to_csv(
    TABLE_DIR
    / "station_parsing_completeness.csv",
    index=False,
)


missing_values = int(
    pds_long[
        "precipitation_depth_mm"
    ].isna().sum()
)


invalid_confidence_order = (
    confidence_table.loc[
        (
            confidence_table[
                "lower_90_mm"
            ]
            > confidence_table[
                "estimate_mm"
            ]
        )
        |
        (
            confidence_table[
                "estimate_mm"
            ]
            > confidence_table[
                "upper_90_mm"
            ]
        )
    ]
    .copy()
)


invalid_confidence_order.to_csv(
    TABLE_DIR
    / "invalid_confidence_interval_order.csv",
    index=False,
)


qc_summary = pd.DataFrame(
    {
        "metric": [
            "raw NOAA files found",
            "stations parsed successfully",
            "stations that failed reparsing",
            "combined long-table rows",
            "unique durations",
            "unique recurrence intervals",
            "estimate types",
            "expected rows per station",
            "stations with complete row counts",
            "missing precipitation values",
            "invalid confidence interval ordering",
        ],
        "value": [
            len(raw_files),
            station_count,
            len(failed_reparses),
            len(pds_long),
            duration_count,
            ari_count,
            estimate_type_count,
            expected_rows_per_station,
            int(
                station_row_counts[
                    "complete"
                ].sum()
            ),
            missing_values,
            len(
                invalid_confidence_order
            ),
        ],
    }
)


qc_output = (
    TABLE_DIR
    / "alabama_noaa_atlas14_pds_qc_summary.csv"
)


qc_summary.to_csv(
    qc_output,
    index=False,
)


print("\nQuality-control summary:")

display(
    qc_summary
)


incomplete_stations = (
    station_row_counts.loc[
        ~station_row_counts[
            "complete"
        ]
    ]
    .copy()
)


if not incomplete_stations.empty:

    print(
        "\nStations with incomplete parsed records:"
    )

    display(
        incomplete_stations
    )


# ============================================================
# 13. FINAL OUTPUTS
# ============================================================

print("\n" + "=" * 72)
print("NOAA ATLAS 14 PDS REPARSE COMPLETED")
print("=" * 72)

print("\nLong-format table:")
print(long_output)

print("\nWide-format table:")
print(wide_output)

print("\nEstimate with confidence bounds:")
print(confidence_output)

print("\nReparse log:")
print(
    TABLE_DIR
    / "reparse_log.csv"
)

print("\nStation completeness:")
print(
    TABLE_DIR
    / "station_parsing_completeness.csv"
)

print("\nQuality-control summary:")
print(qc_output)

Raw NOAA response folder:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\raw_noaa_responses

Station inventory:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\alabama_unique_station_inventory.csv

Unique stations in inventory: 225

Raw NOAA files found: 225

Corrected parser test succeeded.
Rows parsed from 01-0008_pds_frequency.csv: 570
Estimate types: ['estimate', 'lower_90', 'upper_90']
Durations: 19
Recurrence intervals: 10


,station_id,station_name,state,latitude,longitude,noaa_product,atlas_version,noaa_data_type,time_series_type,project_area,noaa_station_name,response_latitude,response_longitude,series,estimate_type,duration,duration_hours,ari_years,precipitation_depth_mm
0,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,1.0,13.0
1,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,2.0,15.0
2,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,5.0,18.0
3,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,10.0,21.0
4,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,25.0,24.0
5,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,50.0,26.0
6,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,100.0,29.0
7,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,200.0,31.0
8,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,500.0,34.0
9,01-0008,ABBEVILLE,AL,31.5703,-85.2483,millimeters,NOAA Atlas 14 Volume 9 Version 2,Precipitation depth,Partial duration,Southeastern States,None,31.5703,-85.2483,pds,estimate,5-min,0.083333,1000.0,37.0


Reparsing existing NOAA files: 100%|██████████| 225/225 [00:08<00:00, 25.83it/s]


Reparse status summary:


,status,stations
0,reparsed_successfully,225


...


Estimate and confidence-bound table:


,station_id,station_name,state,latitude,longitude,duration,duration_hours,ari_years,estimate_mm,lower_90_mm,upper_90_mm,confidence_width_mm,lower_difference_mm,upper_difference_mm
0,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,1.0,174.0,146.0,205.0,59.0,28.0,31.0
1,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,2.0,194.0,163.0,229.0,66.0,31.0,35.0
2,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,5.0,231.0,193.0,274.0,81.0,38.0,43.0
3,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,10.0,266.0,222.0,317.0,95.0,44.0,51.0
4,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,25.0,321.0,263.0,403.0,140.0,58.0,82.0
5,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,50.0,369.0,294.0,468.0,174.0,75.0,99.0
6,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,100.0,421.0,325.0,548.0,223.0,96.0,127.0
7,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,200.0,478.0,356.0,640.0,284.0,122.0,162.0
8,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,500.0,561.0,403.0,772.0,369.0,158.0,211.0
9,01-0008,ABBEVILLE,AL,31.5703,-85.2483,10-day,240.000000,1000.0,629.0,438.0,872.0,434.0,191.0,243.0



Quality-control summary:


,metric,value
0,raw NOAA files found,225
1,stations parsed successfully,225
2,stations that failed reparsing,0
3,combined long-table rows,128250
4,unique durations,19
5,unique recurrence intervals,10
6,estimate types,3
7,expected rows per station,570
8,stations with complete row counts,225
9,missing precipitation values,0



NOAA ATLAS 14 PDS REPARSE COMPLETED

Long-format table:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\alabama_noaa_atlas14_pds_estimates_long.csv

Wide-format table:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\alabama_noaa_atlas14_pds_estimates_wide.csv

Estimate with confidence bounds:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\alabama_noaa_atlas14_pds_estimates_with_confidence_bounds.csv

Reparse log:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\reparse_log.csv

Station completeness:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Flooding\Data\ATLAS 14\NOAA_Atlas14_Alabama_PDS\tables\station_parsing_completeness.csv

Quality-control summary:
D:\projstore\Ismail\Dr Koriche\Intense Rainfall and Flash Floo